# УУНиТ · AI-воркер в Colab

Подготовьте сервер по инструкции `deploy/colab/README.md`. Go, PostgreSQL, RabbitMQ и MinIO остаются
на сервере. Ноутбук запускает Ollama и воркер с локальным HTTP-портом для проверки.

Выберите GPU в настройках среды Colab и пройдите ячейки сверху вниз. Для работы
воркера с внешним приложением потребуется подходящая среда Colab.
[Ограничения и срок жизни среды](https://research.google.com/colaboratory/faq.html).

### 1. Адрес VPS и порты

Укажите публичный IPv4 или DNS-имя VPS, внешние порты и имена пользователей из `.env`.
Добавьте в Colab Secrets `VPS_RABBITMQ_PASSWORD` и `VPS_RAG_POSTGRES_PASSWORD`
и разрешите ноутбуку доступ к ним.

In [ ]:
REPO_DIR = "/content/neti-next-hackaton"

VPS_HOST = "203.0.113.10"
VPS_RABBITMQ_PORT = 25673
VPS_POSTGRES_PORT = 15432
RABBITMQ_USER = "university"
RABBITMQ_REQUEST_QUEUE = "agent_requests"
RABBITMQ_RESPONSE_QUEUE = "agent_responses"
POSTGRES_DB = "university"
RAG_POSTGRES_USER = "rag_app"

OLLAMA_MODEL = "qwen3:4b"
OLLAMA_EMBEDDING_MODEL = "qwen3-embedding:0.6b"
OLLAMA_NUM_CTX = 8192

LOCAL_OLLAMA_PORT = 11434
LOCAL_WORKER_PORT = 8000

### 2. Установите зависимости из репозитория

Ноутбук использует каталог `ai-service` из уже склонированного репозитория.
Если клон лежит по другому пути, укажите его в `REPO_DIR` в первой ячейке.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from urllib.error import URLError
from urllib.parse import quote
from urllib.request import urlopen

from google.colab import userdata

if sys.version_info < (3, 12):
    raise RuntimeError("Выберите в Colab среду с Python 3.12 или новее.")

APP_DIR = Path(REPO_DIR).expanduser().resolve() / "ai-service"
if not (APP_DIR / "pyproject.toml").is_file():
    raise FileNotFoundError(
        f"Не найден {APP_DIR / 'pyproject.toml'}. Укажите путь к клону в REPO_DIR."
    )

if "WORK_DIR" not in globals():
    WORK_DIR = Path(tempfile.mkdtemp(prefix="university-colab-", dir="/content"))
LOG_DIR = WORK_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(APP_DIR)], check=True)

if shutil.which("ollama") is None:
    installer = WORK_DIR / "install-ollama.sh"
    subprocess.run(
        ["curl", "-fSL", "--retry", "3", "https://ollama.com/install.sh", "-o", str(installer)],
        check=True,
    )
    subprocess.run(["sh", str(installer)], check=True)

if shutil.which("nvidia-smi") is not None:
    subprocess.run(["nvidia-smi", "-L"], check=False)

jobs = globals().get("jobs", {})


def stop_job(name):
    process = jobs.get(name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()


def start_job(name, args, *, env=None, cwd=None):
    stop_job(name)
    with (LOG_DIR / f"{name}.log").open("w") as log:
        jobs[name] = subprocess.Popen(
            args, stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT,
            env=env, cwd=cwd,
        )


def wait_until(name, probe, timeout=45):
    until = time.monotonic() + timeout
    while time.monotonic() < until:
        if jobs[name].poll() is not None:
            break
        try:
            if probe():
                return
        except (OSError, URLError):
            pass
        time.sleep(1)
    raise RuntimeError(f"{name} не готов. Откройте ячейку «Диагностика».")


def http_ready(port, path):
    with urlopen(f"http://127.0.0.1:{port}{path}", timeout=2) as response:
        return response.status == 200

### 3. Проверьте соединение с VPS

Ноутбук подключается к внешним портам PostgreSQL и RabbitMQ и применяет миграции.

In [ ]:
if VPS_HOST == "203.0.113.10" or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9.-]*", VPS_HOST):
    raise ValueError("Укажите IPv4 или DNS-имя VPS.")
worker_env = {
    **os.environ,
    "RABBITMQ_URL": (
        f"amqp://{quote(RABBITMQ_USER, safe='')}:"
        f"{quote(userdata.get('VPS_RABBITMQ_PASSWORD'), safe='')}@{VPS_HOST}:{VPS_RABBITMQ_PORT}/"
        "?heartbeat=30"
    ),
    "RABBITMQ_QUEUE": RABBITMQ_REQUEST_QUEUE,
    "RABBITMQ_RESPONSE_QUEUE": RABBITMQ_RESPONSE_QUEUE,
    "RAG_POSTGRES_HOST": VPS_HOST,
    "RAG_POSTGRES_PORT": str(VPS_POSTGRES_PORT),
    "RAG_POSTGRES_DB": POSTGRES_DB,
    "RAG_POSTGRES_USER": RAG_POSTGRES_USER,
    "RAG_POSTGRES_PASSWORD": userdata.get("VPS_RAG_POSTGRES_PASSWORD"),
    "OLLAMA_BASE_URL": f"http://127.0.0.1:{LOCAL_OLLAMA_PORT}",
    "OLLAMA_MODEL": OLLAMA_MODEL,
    "OLLAMA_EMBEDDING_MODEL": OLLAMA_EMBEDDING_MODEL,
    "OLLAMA_NUM_CTX": str(OLLAMA_NUM_CTX),
}

subprocess.run(
    [sys.executable, "-m", "app.index_campus", "--import-only"],
    cwd=APP_DIR, env=worker_env, check=True,
)
rabbit_probe = """
import asyncio
import os

import aio_pika


async def main():
    connection = await aio_pika.connect(os.environ["RABBITMQ_URL"], timeout=10)
    await connection.close()


asyncio.run(main())
"""
subprocess.run([sys.executable, "-c", rabbit_probe], cwd=APP_DIR, env=worker_env, check=True)
print("PostgreSQL и RabbitMQ доступны. Миграции применены.")

### 4. Ollama и индексация

Модели сохраняются в текущей среде Colab. Векторы сохраняются в PostgreSQL на VPS.
Повторный запуск пропускает уже индексированные записи.

In [ ]:
ollama_env = {**os.environ, "OLLAMA_HOST": f"127.0.0.1:{LOCAL_OLLAMA_PORT}"}
if "ollama" not in jobs or jobs["ollama"].poll() is not None:
    start_job("ollama", ["ollama", "serve"], env=ollama_env)
wait_until("ollama", lambda: http_ready(LOCAL_OLLAMA_PORT, "/api/tags"))

for model in (OLLAMA_MODEL, OLLAMA_EMBEDDING_MODEL):
    subprocess.run(["ollama", "pull", model], env=ollama_env, check=True)

subprocess.run(
    [sys.executable, "-m", "app.index_campus"],
    cwd=APP_DIR, env=worker_env, check=True,
)

### 5. Запуск AI-воркера

При успехе `/health` показывает готовность RabbitMQ и Ollama. Теперь можно отправить
запрос через Go API на VPS. Эту ячейку можно повторить для перезапуска воркера.

In [ ]:
start_job(
    "worker",
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1",
     "--port", str(LOCAL_WORKER_PORT)],
    env=worker_env, cwd=APP_DIR,
)
wait_until("worker", lambda: http_ready(LOCAL_WORKER_PORT, "/health"))
with urlopen(f"http://127.0.0.1:{LOCAL_WORKER_PORT}/health", timeout=3) as health:
    print(json.load(health))

### Диагностика и остановка

При сбое подключения проверьте адрес VPS, опубликованные порты и пароли.
Журнал воркера показывает состояние RabbitMQ и ошибки обработки запросов.

In [ ]:
name = "worker"  # ollama или worker
log_file = LOG_DIR / f"{name}.log"
if log_file.is_file():
    print("\n".join(log_file.read_text(errors="replace").splitlines()[-40:]))

In [ ]:
# По окончании сеанса измените значение на True и выполните эту ячейку.
STOP_WORKER = False
if STOP_WORKER:
    for name in ("worker", "ollama"):
        stop_job(name)
    print("Процессы Colab остановлены.")